Step 2: Data preparation
- Load Breast Cancer Wisconsin (Diagnostic)
- Hold out a global test set (never seen by any hospital)
- Fit the scaler on the training part only, then scale
- Split the remaining training data across 3 isolated hospitals
Outputs: breast_cancer.csv, global_test.csv, hospital_1..3.csv, scaler.pkl


<div align="center">

# Breast Cancer Dataset
### Exploratory Data Analysis (EDA)

<img src="https://commons.wikimedia.org/wiki/Special:FilePath/Histopathology_of_invasive_ductal_carcinoma_of_the_breast.jpg"
     width="700"
     alt="Histopathology of breast cancer">

*Image source: Wikimedia Commons — CC0 License*

</div>


In [11]:
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ------------------------- Config -------------------------
try:
    FOLDER = os.path.dirname(os.path.abspath(__file__))
except NameError:
    FOLDER = os.getcwd()
NUM_HOSPITALS = 3
TEST_SIZE = 0.2
PARTITION = "iid"        # "iid" or "non_iid"
DIRICHLET_ALPHA = 0.5
SEED = 42
TARGET = "Diagnosis"     # 1 = Malignant, 0 = Benign


# ------------------------- Helpers -------------------------
def split_iid(n_samples, n_parts, rng):
    idx = rng.permutation(n_samples)
    return [part.tolist() for part in np.array_split(idx, n_parts)]


def split_non_iid(y, n_parts, alpha, rng, min_size=15):
    while True:
        parts = [[] for _ in range(n_parts)]
        for c in np.unique(y):
            idx = np.where(y == c)[0]
            rng.shuffle(idx)
            props = rng.dirichlet([alpha] * n_parts)
            cuts = (np.cumsum(props) * len(idx)).astype(int)[:-1]
            for i, chunk in enumerate(np.split(idx, cuts)):
                parts[i] += chunk.tolist()
        if min(len(p) for p in parts) >= min_size:
            return parts


# ------------------------- Main -------------------------
def main():
    rng = np.random.default_rng(SEED)

    # 1. Load
    data = load_breast_cancer(as_frame=True)
    features = list(data.feature_names)
    df = data.frame.copy()
    df[TARGET] = 1 - df.pop("target")
    df.to_csv(os.path.join(FOLDER, "breast_cancer.csv"), index=False)

    # 2. Global test set
    train_df, test_df = train_test_split(
        df, test_size=TEST_SIZE, random_state=SEED, stratify=df[TARGET]
    )
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    # 3. Scaling
    scaler = StandardScaler().fit(train_df[features])
    train_df[features] = scaler.transform(train_df[features])
    test_df[features] = scaler.transform(test_df[features])
    joblib.dump(scaler, os.path.join(FOLDER, "scaler.pkl"))
    test_df.to_csv(os.path.join(FOLDER, "global_test.csv"), index=False)

    # 4. Split across hospitals
    if PARTITION == "iid":
        parts = split_iid(len(train_df), NUM_HOSPITALS, rng)
    else:
        parts = split_non_iid(train_df[TARGET].values, NUM_HOSPITALS, DIRICHLET_ALPHA, rng)

    # 5. Save one CSV per hospital
    print("✅ Data split done. Files created in:", FOLDER)
    for i, idx in enumerate(parts, 1):
        hospital_df = train_df.iloc[idx]
        hospital_df.to_csv(os.path.join(FOLDER, f"hospital_{i}.csv"), index=False)
        print(f"🏥 Hospital {i}: {len(hospital_df)} patients "
              f"(malignant = {hospital_df[TARGET].mean():.0%})")
    print(f"🧪 Global test: {len(test_df)} patients (malignant = {test_df[TARGET].mean():.0%})")


if __name__ == "__main__":
    main()

✅ Data split done. Files created in: c:\Users\dell\AppData\Local\Programs\Microsoft VS Code
🏥 Hospital 1: 152 patients (malignant = 41%)
🏥 Hospital 2: 152 patients (malignant = 41%)
🏥 Hospital 3: 151 patients (malignant = 30%)
🧪 Global test: 114 patients (malignant = 37%)
